<a href="https://colab.research.google.com/github/kumarpal1107/pythonbasics/blob/main/Detectron2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#1
Detectron2 is an open-source computer vision framework developed by Facebook AI Research (FAIR) for object detection, segmentation, and other visual tasks.

It differs from previous frameworks primarily by being built on PyTorch and featuring a more modular, flexible design, and faster performance

In [ ]:
#2
Data annotation in Detectron2 involves labeling custom images (using tools like CVAT or Roboflow) to define bounding boxes or segmentation masks, which are then exported into the COCO JSON format to map class IDs to object locations.

This process requires registering the dataset within Detectron2, ensuring high-quality, consistent labels that allow the model to learn specific features, such as shapes or categories, beyond standard pre-trained datasets.

The importance of this step cannot be overstated; accurate annotations are essential for achieving high precision in object detection, as they serve as the ground truth that enables supervised learning. Proper annotation directly impacts model performance, reduces training time, and ensures the model can correctly identify, classify, and segment target objects in new, unannotated images.

In [ ]:
#3
To train a custom object detection model using Detectron2, the general steps involve preparing a custom dataset in a compatible format (like COCO), configuring the necessary data loaders and evaluation metrics, setting up a model configuration file (often starting from a pre-trained model checkpoint), initializing a DefaultTrainer instance, and finally, calling the train method.

This process leverages Detectron2's high-level API to automate much of the standard training loop, data augmentation, and logging.

In [ ]:
#4
In Detectron2, evaluation curves primarily refer to Precision-Recall (PR) curves, which help visualize a model's trade-off between precision and recall at various confidence thresholds.

The performance is summarized using key metrics like mAP (mean Average Precision) and IoU (Intersection over Union), which quantify the accuracy of object localization and classification

In [ ]:
#5
Detectron2 (PyTorch, Meta AI) excels in flexibility, advanced tasks (panoptic/instance segmentation, keypoint detection) with a modular design but has a steeper learning curve and high GPU needs, while TFOD2 (TensorFlow, Google) offers broader ecosystem integration, simpler integration into Google Cloud, often easier entry for beginners, and strong production readiness, though potentially less cutting-edge on pure research models, both providing high performance depending on model choice, with TFOD2 potentially having better market adoption.

In [1]:
#6
import torch, detectron2
!nvcc --version
TORCH_VERSION = ".".join(torch.__version__.split(".")[:2])
CUDA_VERSION = torch.__version__.split("+")[-1]
print("torch: ", TORCH_VERSION, "; cuda: ", CUDA_VERSION)
print("detectron2:", detectron2.__version__)

/bin/bash: line 1: nvcc: command not found
torch:  2.9 ; cuda:  cpu
detectron2: 0.6

import detectron2
from detectron2.utils.logger import setup_logger
setup_logger()

# import some common libraries
import numpy as np
import os, json, cv2, random
from google.colab.patches import cv2_imshow

# import some common detectron2 utilities
from detectron2 import model_zoo
from detectron2.engine import DefaultPredictor
from detectron2.config import get_cfg
from detectron2.utils.visualizer import Visualizer
from detectron2.data import MetadataCatalog, DatasetCatalog

SyntaxError: invalid syntax (2418480692.py, line 9)

In [2]:
#8
import os
from detectron2.engine import DefaultTrainer
from detectron2.config import get_cfg
from detectron2 import model_zoo

def setup_detectron2_training_config(config_file_path, pretrained_weights_url, output_dir="./output_train"):
    """
    Sets up the Detectron2 configuration for training using pretrained weights.

    Args:
        config_file_path (str): The path to the base YAML configuration file in the model zoo.
        pretrained_weights_url (str): The URL or shortcut for the pretrained model weights.
        output_dir (str): The directory where training outputs (checkpoints, logs) will be saved.

    Returns:
        cfg (CfgNode): The configured Detectron2 configuration object.
    """
    # 1. Get a fresh configuration
    cfg = get_cfg()

    # 2. Merge with a pre-trained model's configuration from the model zoo
    # This automatically sets many defaults, including the model architecture
    cfg.merge_from_file(model_zoo.get_config_file(config_file_path))

    # 3. Set the pre-trained weights
    # Detectron2 will download these weights automatically if not already cached
    cfg.MODEL.WEIGHTS = pretrained_weights_url

    # 4. Configure paths for training
    # Set the output directory for logs and saved model checkpoints
    cfg.OUTPUT_DIR = output_dir
    os.makedirs(cfg.OUTPUT_DIR, exist_ok=True) # Create the output directory if it doesn't exist

    # 5. Customize other training parameters (example values for fine-tuning)
    # Note: Adjust these based on your specific dataset and hardware (e.g., GPU VRAM)
    cfg.DATASETS.TRAIN = ("your_custom_dataset_train",) # Register your custom training dataset name(s) here
    cfg.DATASETS.TEST = ("your_custom_dataset_val",)   # Register your custom validation dataset name(s) here
    cfg.DATALOADER.NUM_WORKERS = 2                     # Number of data loading threads
    cfg.SOLVER.IMS_PER_BATCH = 2                       # Batch size (adjust based on GPU memory)
    cfg.SOLVER.BASE_LR = 0.00025                       # Initial learning rate for fine-tuning
    cfg.SOLVER.MAX_ITER = 300                          # Maximum number of iterations for training
    cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 128     # Number of regions of interest per image
    cfg.MODEL.ROI_HEADS.NUM_CLASSES = 1                # Number of classes in your custom dataset (e.g., 1 for "balloon")

    return cfg

# Example usage in a training script:
if __name__ == "__main__":
    # Define model details from the Detectron2 Model Zoo
    # The URL shortcut 'detectron2://...' is preferred for automatic downloading
    CONFIG_FILE = "COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"
    WEIGHTS_URL = "detectron2://COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x/137849600/model_final_f10217.pkl"
    OUTPUT_FOLDER = "./custom_train_output"

    # Configure the model
    cfg = setup_detectron2_training_config(CONFIG_FILE, WEIGHTS_URL, OUTPUT_FOLDER)

    print(f"Configuration setup complete. Weights path: {cfg.MODEL.WEIGHTS}")
    print(f"Output directory: {cfg.OUTPUT_DIR}")
    print("To start training, you would typically register your custom datasets first (see cfg.DATASETS.TRAIN), then use DefaultTrainer(cfg).train().")

    # Example of how to use the config to start training (requires registered datasets)
    # trainer = DefaultTrainer(cfg)
    # trainer.resume_or_load(resume=False)
    # trainer.train()

ModuleNotFoundError: No module named 'detectron2'

In [ ]:
#9

import cv2
import os
from detectron2.config import get_cfg
from detectron2.engine import DefaultPredictor
from detectron2 import model_zoo
from detectron2.utils.visualizer import Visualizer
from detectron2.data import MetadataCatalog

# 1. Setup configuration
cfg = get_cfg()
# Use the same config file used during training
CONFIG_FILE = "COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"
cfg.merge_from_file(model_zoo.get_config_file(CONFIG_FILE))

# 2. Load pre-trained weights and set threshold
# For demonstration, use pre-trained weights from the model zoo
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(CONFIG_FILE)
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.7  # Set threshold for this model
cfg.MODEL.DEVICE = "cpu"  # Use "cpu" if no GPU is available

# 3. Initialize predictor
predictor = DefaultPredictor(cfg)

# 4. Load image and run inference
# Create a dummy image for demonstration if 'new_image.jpg' does not exist
if not os.path.exists("./new_image.jpg"):
    dummy_image = np.zeros((400, 600, 3), dtype=np.uint8)
    cv2.putText(dummy_image, "Dummy Image", (50, 200), cv2.FONT_HERSHEY_SIMPLEX, 2, (255, 255, 255), 3)
    cv2.imwrite("./new_image.jpg", dummy_image)

im = cv2.imread("./new_image.jpg")
outputs = predictor(im)

# 5. Visualize and display
# For visualization without a custom dataset, use a generic COCO metadata
v = Visualizer(im[:, :, ::-1], metadata=MetadataCatalog.get("coco_2017_val"), scale=1.2)
out = v.draw_instance_predictions(outputs["instances"].to("cpu"))
cv2_imshow(out.get_image()[:, :, ::-1])
# cv2.waitKey(0) # Not needed with cv2_imshow

In [ ]:
#10
To build a wildlife monitoring system with Detectron2, the pipeline begins with collecting diverse image data from camera traps, followed by labeling with tools like CVAT to create COCO-format annotations.

The training phase involves selecting a pre-trained backbone (e.g., ResNet-50 or Swin Transformer) from the Detectron2 Model Zoo and fine-tuning it on the forest dataset. For deployment, the model is exported to TorchScript or ONNX for optimized inference on edge devices like NVIDIA Jetson.

To handle occlusion, use heavy data augmentation (Random Erasing or Crop) and temporal consistency (tracking across frames). For nighttime detection, incorporate infrared-specific training data and apply image enhancement techniques like CLAHE to improve contrast in low-light conditions.